## Notebook Overview

The following notebook provides a practical walkthrough of the CRUD operations on datasets that the CVDMS programmatic API allows.
Before running the examples, ensure that the **CDK application has been deployed** to your AWS account.
The code assumes the images described in the *selection configuration* below are present in the CVDMS system.

---

## Supported Task Types (*label_type*)

CVDMS currently supports five computer vision task types:

1. **Single Label Classification** (`label_type = "single-label"`)
2. **Multi-Label Classification** (`label_type = "multi-label"`)
3. **Object Detection** (`label_type = "object-detection"`)
4. **Semantic Segmentation** (`label_type = "semantic-segmentation"`)
5. **Instance Segmentation** (`label_type = "instance-segmentation"`)

In the examples below, one of these task types is selected for the dataset, and thenceforth will
be tied to that globally unique *dataset_id*.

---

## Purpose of This Notebook

These examples are **not training workflows**. Their purpose is to demonstrate how imagery and labels previously ingested into the CVDMS infrastructure
can be used to form dataset objects.

Running this notebook allows you to:

- confirm the dataset operations are behaving as expected
- test the creation of datasets of different label types
- create manifests for a dataset and assign splits for training, validation, and testing
- retrieve the latest dataset metadata
- create a new dataset version through the update flow
- delete the dataset and all of its versions when testing is complete

### Authenticate with AWS

Before interacting with the deployed infrastructure, you must authenticate with AWS and obtain temporary credentials.

The code cell below performs an AWS SSO login using the profile configured on your local machine.

Ensure that:

- You have created a user in **AWS IAM Identity Center (SSO)**.
- The user is assigned to the AWS account where the **CDK application was deployed**.
- The assigned role has permissions to access the CVDMS infrastructure.
- The profile you specify exists in your local AWS configuration (`~/.aws/config`) and points to the correct **account, role, and region**.

Running the command will open a browser window for authentication if your credentials are not already cached.

- `app_name` – the name used when deploying the CDK application
- `profile_name` – the AWS CLI profile used for authentication


In [ ]:
# This is the name of the CDK app you deployed.
app_name = "cvdmsv1"

# This is your profile name (see the local aws config file).
profile_name = "developers_admin"

# Login to AWS. This will redirect you to a login screen or be approved if credentials are still valid from your previous login.
!aws sso login --profile {profile_name}

### Initialize the CVDMS Client

The code below imports the programmatic API entry point and creates a client instance used to interact with the CVDMS infrastructure.

The client uses the AWS profile you authenticated with in the previous step and loads the necessary configuration (such as bucket names and table names) from AWS Systems Manager Parameter Store.


In [ ]:
# Instantiate the client.
from cvdms_platform import CvdmsApp

app = CvdmsApp(app_name=app_name,
               profile_name=profile_name)

### Create a Dataset

The code below creates a new dataset given the necessary parameters.

The `submit_create_dataset()` method creates the dataset and instantiates version 1 according to the selection criteria specified.

Parameters used in this example:

- **`dataset_id`**  
  This is a string, human-readable, and globally unique dataset id.

- **`label_type`**  
  Specifies the task type for the dataset. It cannot be changed later in update, and must be one of the supported values:
  `single-label`, `multi-label`, `object-detection`, `semantic-segmentation`, or `instance-segmentation`.

- **`description`**  
  A descriptive string indicating briefly what the dataset is. Can be at most 500 characters.

- **`selection_config`**  
  A dictionary specifying allowable image features when forming the dataset. This makes up the selection criteria for candidate imagery
  to include in version 1 of the dataset. The key `"allowed_classes"` is always required.

- **`split_strategy_name`**  
  A string name of the strategy used to assign splits. Currently, only one technique is implemented, and is utilized by
  setting this variable to `"stratified_v1"`.


- **`honor_source_splits`**
  A boolean True or False. This indicates if the original source data's splits, if present during the ingestion step, are to be preserved in the dataset's
  created manifests.

Running this call starts the dataset creation process.

In [ ]:
dataset_id = "eurosat-single-label-test-id"
label_type = "single-label"
description = "Single-label EuroSAT test dataset with forest, highway, and annualcrop classes"

selection_config = {
    "allowed_classes": ["forest", "highway", "annualcrop"],
    "allowed_sources": ["eurosat"]
}

split_strategy_name = "stratified_v1"
honor_source_splits = False

creation_out = app.submit_create_dataset(
    dataset_id=dataset_id,
    label_type=label_type,
    description=description,
    selection_config=selection_config,
    split_strategy_name=split_strategy_name,
    honor_source_splits=honor_source_splits
)

### Check Submission Status

Run this cell to check that the `submit_create_dataset` call was successful. The work takes place inside a step function and branches depending on the task that was selected. In this case, the 'create' branch will run and create the dataset and all pertinent files and table rows behind the scenes. The above call will either raise, return an error field in a dictionary, or a job id on success.

In [ ]:
job_id = creation_out.get("job_id")
creation_error = creation_out.get("error")

if job_id:
    print(f"Dataset creation job submitted successfully. Job ID: {job_id}")
else:
    print(f"Dataset creation job submission failed. Error: {creation_error}")
    print("See local API logs in cvdms_platform/api_logs for more details.")

### Retrieve a Dataset's Information

The cell below fetches the latest dataset information using the local client function get_dataset
This is useful for confirming the latest version metadata, label type, split strategy, and S3 artifact pointers.


In [ ]:
get_out = app.get_dataset(dataset_id=dataset_id)
print(get_out)

### Update the Dataset to Create a New Version

The update flow creates a **new dataset version** rather than editing the old one in place.

You can use:

- `operation="add"` to add imagery matching a new selection config
- `operation="remove"` to remove imagery matching a new selection config

You can also choose between two split behaviors:

- **`split_approach="maintain"`**  
  Preserve existing split assignments where possible and reuse the current split strategy automatically. When `split_approach = "maintain"`,
  `split_strategy_name` must be set to `None` because the prior strategy (_latest version_) is always reused.

- **`split_approach="rebalance"`**  
  Recompute the dataset split assignment and explicitly provide a split strategy name. When `split_approach = "rebalance"`, `split_strategy_name`
  must not be `None`.

The allowed classes indicated in the create call is immutable. That is, if you want to add or remove classes, you must create a new dataset. The
`allowed_classes` field during update must be a _subset_ of the original `allowed_classes` field used when creating the dataset.

Similarly, `label_type` and `honor_source_splits` are immutable and cannot be changed with an update call.

In [ ]:
dataset_id = "eurosat-single-label-test-id"
operation = "remove" # "add" or "remove"

selection_config = {
    "allowed_classes": ["forest"],
    "allowed_sources": ["eurosat"]
}

split_approach = "maintain" # "maintain" or "rebalance"
split_strategy_name = None # None if split_approach is maintain, otherwise a strategy string name is required.
description = "Updating to v2 by removing forest images from EuroSAT source."

update_out = app.submit_update_dataset(dataset_id=dataset_id,
                                       operation=operation,
                                       selection_config=selection_config,
                                       split_approach=split_approach,
                                       split_strategy_name=split_strategy_name,
                                       description=description)
print(update_out)

### Check Updated Dataset State

After running the update flow, retrieve the dataset again to confirm that the latest version changed
and that the newest metadata now points at the newly created version.

This is one of the most useful checks while testing, since it verifies the read path agrees with the update path.


In [ ]:
import json
get_out_after_update = app.get_dataset(dataset_id=dataset_id)
print(json.dumps(get_out_after_update, indent = 2))

### Delete the Dataset and All Versions

The code below deletes the dataset and all of its versions.

The current delete flow is best-effort and is intended to remove:

1. all Iceberg membership rows for the dataset
2. all S3 artifacts under the dataset prefix
3. the dataset row and all dataset-version rows in DynamoDB

Because this is destructive, only run the cell below when you are ready to clean up the dataset created for testing.


In [ ]:
delete_out = app.submit_delete_dataset_all_versions(dataset_id=dataset_id)
print(delete_out)

### Confirm Deletion

Run this final cell to confirm the dataset no longer exists through the read path.

If deletion completed fully, the returned dictionary should indicate that the dataset does not exist.


In [ ]:
post_delete_out = app.get_dataset(dataset_id=dataset_id)
print(post_delete_out)

In practice, the most common reason for a failure in this notebook is that the `selection_config`
does not match data that currently exists in the canonical tables. When testing, it is normal to
adjust the selection criteria a few times until the create and update flows target the exact imagery you want.